# 02 · 向量化 Vectorization

> **能力标签**：SH3（NumPy 与向量化计算 / NumPy & vectorized computing）

## 目标 Objectives

完成本课后，你应该能够：

1. 解释**向量化（vectorization）**的含义：用 NumPy 内置操作取代 Python 循环。
2. 用 `time.perf_counter` 对比循环版与向量化版的耗时，理解性能差距。
3. 实现 z-score 标准化（standardize）的向量化版本。
4. 区分**ufunc**（逐元素函数）与普通 Python 函数的行为。

> 对应能力：**SH3**


## 概念讲解 Concepts

### 什么是向量化？

向量化是指：把"逐元素"的计算从 Python 层面下移到 NumPy 的 C/Fortran 层，
利用 SIMD 指令和编译器优化，实现**批量计算**。

```
纯 Python 循环：  for x in data: result.append(x * 2)
向量化：           result = data * 2          # NumPy
```

Python 解释器每次循环迭代都有**固定开销**（GIL 检查、对象创建、引用计数）。
NumPy 把这些开销摊薄到整个数组，因此数组越大，加速比越高。

---

### ufunc（通用函数 Universal Function）

NumPy 的 ufunc 支持**逐元素**操作，并遵守广播规则：

| 操作 | 示例 |
|------|------|
| 加减乘除 | `a + b`、`a * 2` |
| 数学函数 | `np.exp(a)`、`np.sqrt(a)` |
| 比较 | `a > 0`、`a == b` |

---

### z-score 标准化 Standardize

给定一维数组 $x$，z-score 标准化定义为：

$$z_i = \frac{x_i - \bar{x}}{\sigma_x}$$

其中 $\bar{x}$ 是均值，$\sigma_x$ 是标准差。

向量化实现：
```python
def standardize(x):
    return (x - x.mean()) / x.std()
```

---

### 计时技巧 Timing

用 `time.perf_counter` 手动计时（比 `%timeit` magic 更显式、更可移植）：

```python
import time
t0 = time.perf_counter()
# ... 你的代码 ...
elapsed = time.perf_counter() - t0
print(f"耗时: {elapsed*1000:.3f} ms")
```


## 示例 Worked Example：标准化（standardize）

In [ ]:
import numpy as np
import time

rng = np.random.default_rng(42)
x = rng.normal(5.0, 2.0, size=100_000)   # 10 万个正态随机数

# ── 版本 A：Python 循环 ────────────────────────────────────────────────────
def standardize_loop(x):
    n = len(x)
    mu = sum(x) / n
    sigma = (sum((xi - mu)**2 for xi in x) / n) ** 0.5
    return [(xi - mu) / sigma for xi in x]

# ── 版本 B：NumPy 向量化 ───────────────────────────────────────────────────
def standardize(x: np.ndarray) -> np.ndarray:
    """z-score 标准化（vectorized）。"""
    return (x - x.mean()) / x.std()

# ── 计时对比 ──────────────────────────────────────────────────────────────
# 注意：对 10 万元素用循环很慢，这里只用 1000 元素对比
x_small = x[:1000]

t0 = time.perf_counter()
for _ in range(10):
    _ = standardize_loop(x_small.tolist())
loop_ms = (time.perf_counter() - t0) / 10 * 1000

t0 = time.perf_counter()
for _ in range(10):
    _ = standardize(x_small)
vec_ms = (time.perf_counter() - t0) / 10 * 1000

print(f"循环版 (1000 元素): {loop_ms:.3f} ms")
print(f"向量化 (1000 元素): {vec_ms:.4f} ms")
print(f"加速比: {loop_ms/vec_ms:.1f}x")

# ── 验证结果一致 ──────────────────────────────────────────────────────────
z = standardize(x)
print(f"\n均值 (应≈0): {z.mean():.2e}")
print(f"标准差 (应≈1): {z.std():.6f}")


In [ ]:
# 向量化的其他常见模式
import numpy as np

rng = np.random.default_rng(0)
A = rng.integers(0, 100, size=(5, 4))

print("原始矩阵 A:\n", A)

# 行求和（sum over columns）
row_sums = A.sum(axis=1)    # axis=1 = 沿列方向折叠 → 每行一个数
print("\n每行之和:", row_sums)

# 列最大值
col_max = A.max(axis=0)
print("每列最大值:", col_max)

# 累积求和（cumulative sum）
print("\n第 0 行的累积和:", np.cumsum(A[0]))

# 条件运算（逐元素 ufunc）
print("\nA > 50 的元素个数:", (A > 50).sum())
print("A 中大于 50 的元素:", A[A > 50])


## 动手 Exercise

实现 `clip_and_normalize`：先将数组元素截断到 `[lo, hi]`（用 `np.clip`），再做 z-score 标准化。
要求全程向量化，不使用任何 Python 循环。


In [ ]:
import numpy as np

def clip_and_normalize(x: np.ndarray, lo: float, hi: float) -> np.ndarray:
    """截断后 z-score 标准化（vectorized，无循环）。"""
    clipped = np.clip(x, lo, hi)
    return (clipped - clipped.mean()) / clipped.std()

# 验证
rng = np.random.default_rng(7)
x = rng.normal(0, 3, size=10)
result = clip_and_normalize(x, -2.0, 2.0)
print("输入:", x.round(2))
print("结果:", result.round(4))
print("均值≈0:", abs(result.mean()) < 1e-10)
print("std≈1:", abs(result.std() - 1.0) < 1e-10)


## 延伸阅读 Further Reading

1. **NumPy 官方文档 — ufuncs**: <https://numpy.org/doc/stable/reference/ufuncs.html>
2. **NumPy for MATLAB users**: <https://numpy.org/doc/stable/user/numpy-for-matlab-users.html>
3. **《High Performance Python》第 6 章** — 矩阵与向量的并行化
4. **Jake VanderPlas — Python Data Science Handbook, Ch.2** （免费在线）


## 关联练习 Related Assignments

本课对应练习包：**`w2-numpy`**（向量化基础题目包含 `standardize` 与 `pairwise_sq_dists`）

```bash
python check.py w2-numpy
```

> 能力标签：**SH3** · threshold ≥ 0.7
